In [ ]:
import altair as alt
import polars as pl
from pathlib import Path

DATA_PATH_CANDIDATES = [
    Path("footbench/outputs/data/pairwise_results.csv"),
    Path("outputs/data/pairwise_results.csv"),
    Path("data/pairwise_results.csv"),
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), DATA_PATH_CANDIDATES[0])

SITE_BLUE = "#2563eb"
TEXT_COLOR = "#111827"
MUTED_TEXT = "#4b5563"
GRID_COLOR = "#e5e7eb"
MODEL_PALETTE = [
    "#2563eb", "#0891b2", "#16a34a", "#ca8a04", "#dc2626", "#9333ea",
    "#0f766e", "#ea580c", "#4f46e5", "#be123c", "#64748b", "#7c3aed",
]
MODEL_LABELS = {
    "claude-fable-5": "Claude Fable 5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-opus-4-8": "Claude Opus 4.8",
    "claude-sonnet-4-6": "Claude Sonnet 4.6",
    "deepseek-v4-flash": "DeepSeek V4 Flash",
    "deepseek-v4-pro": "DeepSeek V4 Pro",
    "gemini-3.1-flash-lite": "Gemini 3.1 Flash-Lite",
    "gemini-3.1-pro-preview": "Gemini 3.1 Pro Preview",
    "gemini-3.5-flash": "Gemini 3.5 Flash",
    "gpt-5.4-mini": "OpenAI GPT 5.4 Mini",
    "gpt-5.5": "OpenAI GPT 5.5",
    "gpt-5.5-pro": "OpenAI GPT 5.5 Pro",
}
COMPANY_ORDER = ["Anthropic", "OpenAI", "Google", "DeepSeek"]
COMPANY_COLORS = ["#2563eb", "#16a34a", "#ea580c", "#0891b2"]
MODEL_COMPANIES = {
    "Claude Fable 5": "Anthropic",
    "Claude Haiku 4.5": "Anthropic",
    "Claude Opus 4.8": "Anthropic",
    "Claude Sonnet 4.6": "Anthropic",
    "DeepSeek V4 Flash": "DeepSeek",
    "DeepSeek V4 Pro": "DeepSeek",
    "Gemini 3.1 Flash-Lite": "Google",
    "Gemini 3.1 Pro Preview": "Google",
    "Gemini 3.5 Flash": "Google",
    "OpenAI GPT 5.4 Mini": "OpenAI",
    "OpenAI GPT 5.5": "OpenAI",
    "OpenAI GPT 5.5 Pro": "OpenAI",
}

alt.data_transformers.disable_max_rows()


@alt.theme.register("footbench", enable=True)
def footbench_theme():
    return alt.theme.ThemeConfig({
        "config": {
            "background": "white",
            "font": "Inter, system-ui, sans-serif",
            "axis": {
                "domain": False,
                "gridColor": GRID_COLOR,
                "labelColor": MUTED_TEXT,
                "labelFontSize": 12,
                "labelLimit": 240,
                "tickColor": GRID_COLOR,
                "titleColor": TEXT_COLOR,
                "titleFontSize": 13,
                "titlePadding": 10,
            },
            "axisX": {"grid": True},
            "axisY": {"grid": False},
            "header": {
                "labelColor": TEXT_COLOR,
                "labelFontSize": 14,
                "labelFontWeight": "bold",
                "title": None,
            },
            "legend": {
                "labelColor": MUTED_TEXT,
                "labelFontSize": 12,
                "symbolSize": 100,
                "titleColor": TEXT_COLOR,
                "titleFontSize": 13,
            },
            "title": {
                "anchor": "start",
                "color": TEXT_COLOR,
                "fontSize": 18,
                "fontWeight": "bold",
                "offset": 18,
            },
            "view": {"stroke": None},
        }
    })


results = (
    pl.read_csv(DATA_PATH)
    .with_columns(
        pl.col("judge", "model_a", "model_b", "criterion", "winner").str.strip_chars()
    )
    .with_columns(
        pl.col("judge", "model_a", "model_b", "winner").replace(MODEL_LABELS)
    )
)

candidate_results = pl.concat(
    [
        results.select("judge", "criterion", pl.col("model_a").alias("candidate"), "winner"),
        results.select("judge", "criterion", pl.col("model_b").alias("candidate"), "winner"),
    ]
).with_columns(
    (pl.col("candidate") == pl.col("winner")).cast(pl.Int8).alias("win"),
    pl.col("candidate").replace(MODEL_COMPANIES).alias("company"),
)

overall_scores = (
    candidate_results.group_by("candidate", "company")
    .agg(pl.len().alias("comparisons"), pl.sum("win").alias("wins"))
    .with_columns((pl.col("wins") / pl.col("comparisons") * 100).alias("win_pct"))
    .sort("win_pct", descending=True)
)

model_order = overall_scores.get_column("candidate").to_list()
judge_models = sorted(results.get_column("judge").unique().to_list())
judge_order = [model for model in model_order if model in judge_models]
criterion_order = sorted(results.get_column("criterion").unique().to_list())

criterion_scores = (
    candidate_results.group_by("criterion", "candidate", "company")
    .agg(pl.len().alias("comparisons"), pl.sum("win").alias("wins"))
    .with_columns((pl.col("wins") / pl.col("comparisons") * 100).alias("win_pct"))
)

judge_model_scores = (
    candidate_results.group_by("judge", "candidate")
    .agg(pl.len().alias("comparisons"), pl.sum("win").alias("wins"))
    .with_columns((pl.col("wins") / pl.col("comparisons") * 100).alias("win_pct"))
)

judge_matrix_scores = (
    candidate_results.filter(pl.col("candidate").is_in(judge_order))
    .group_by("judge", "candidate")
    .agg(pl.len().alias("comparisons"), pl.sum("win").alias("wins"))
    .with_columns((pl.col("wins") / pl.col("comparisons") * 100).alias("win_pct"))
)


# Footbench results

Use `footbench/outputs/data/pairwise_results.csv` for all visualizations.

Use Python and Altair to create all visualizations.

## Overall pairwise results

### Description

Type of chart: Lolipop chart.

x-axis: Percentage from 0 to 100.
y-axis: categorical - each model is one category
What the numbers represent: The win percentage, overall and across both criteria, of each model.

Sort by overall win percentage. Model with the highest win percentage is at the top.

Lines and dots have single color.



In [ ]:
lollipop_data = overall_scores.with_columns(pl.lit(0.0).alias("zero"))
percent_axis = alt.Axis(values=[0, 25, 50, 75, 100], labelExpr="datum.value + '%'")
company_color = alt.Color(
    "company:N",
    sort=COMPANY_ORDER,
    scale=alt.Scale(domain=COMPANY_ORDER, range=COMPANY_COLORS),
    legend=alt.Legend(
        title=None,
        orient="bottom",
        direction="horizontal",
        columns=4,
        symbolType="stroke",
        symbolStrokeWidth=4,
        labelLimit=0,
    ),
)

base = alt.Chart(lollipop_data).encode(
    y=alt.Y("candidate:N", sort=model_order, title=None),
    color=company_color,
    tooltip=[
        alt.Tooltip("company:N", title="Company"),
        alt.Tooltip("candidate:N", title="Model"),
        alt.Tooltip("win_pct:Q", title="Win percentage", format=".1f"),
        alt.Tooltip("wins:Q", title="Wins"),
        alt.Tooltip("comparisons:Q", title="Comparisons"),
    ],
)

rules = base.mark_rule(opacity=0.45, strokeWidth=3).encode(
    x=alt.X("zero:Q", scale=alt.Scale(domain=[0, 100]), axis=percent_axis, title="Overall win percentage"),
    x2="win_pct:Q",
)

points = base.mark_circle(size=95).encode(
    x=alt.X("win_pct:Q", scale=alt.Scale(domain=[0, 100]), axis=percent_axis, title="Overall win percentage"),
)

(rules + points).properties(
    width=720,
    height=30 * len(model_order),
    title="Overall Pairwise Win Percentage",
)


## Pairwise results by criterion

### Description

Same plot as `## Overall pairwise results`, but faceted by `criterion`

In [ ]:
criterion_lollipop_data = criterion_scores.with_columns(pl.lit(0.0).alias("zero"))
percent_axis = alt.Axis(values=[0, 25, 50, 75, 100], labelExpr="datum.value + '%'")
company_color = alt.Color(
    "company:N",
    sort=COMPANY_ORDER,
    scale=alt.Scale(domain=COMPANY_ORDER, range=COMPANY_COLORS),
    legend=alt.Legend(
        title=None,
        orient="bottom",
        direction="horizontal",
        columns=4,
        symbolType="stroke",
        symbolStrokeWidth=4,
        labelLimit=0,
    ),
)

base = alt.Chart(criterion_lollipop_data).encode(
    y=alt.Y("candidate:N", sort=model_order, title=None),
    color=company_color,
    tooltip=[
        alt.Tooltip("criterion:N", title="Criterion"),
        alt.Tooltip("company:N", title="Company"),
        alt.Tooltip("candidate:N", title="Model"),
        alt.Tooltip("win_pct:Q", title="Win percentage", format=".1f"),
        alt.Tooltip("wins:Q", title="Wins"),
        alt.Tooltip("comparisons:Q", title="Comparisons"),
    ],
)

rules = base.mark_rule(opacity=0.45, strokeWidth=3).encode(
    x=alt.X("zero:Q", scale=alt.Scale(domain=[0, 100]), axis=percent_axis, title="Win percentage"),
    x2="win_pct:Q",
)

points = base.mark_circle(size=80).encode(
    x=alt.X("win_pct:Q", scale=alt.Scale(domain=[0, 100]), axis=percent_axis, title="Win percentage"),
)

alt.layer(rules, points).properties(
    width=340,
    height=30 * len(model_order),
).facet(
    column=alt.Column("criterion:N", sort=criterion_order, title=None),
).properties(
    title="Pairwise Win Percentage by Criterion",
)


## Pairwise results by model

### Description

Line chart

x-axis: categorical - each judge model
y-axis: overall win percentage

Lines are linked / grouped by candidate model
Therefore, each line represents a single candidate model and that model's overall score
for each jduge
Lines have distinct color

In [ ]:
line_hover = alt.selection_point(
    name="line_hover",
    fields=["candidate"],
    on="pointerover",
    clear="pointerout",
    empty=False,
)
line_click = alt.selection_point(
    name="line_click",
    fields=["candidate"],
    on="click",
    empty=False,
)

active_candidate = " || ".join(
    [
        "!length(data('line_hover_store')) && !length(data('line_click_store'))",
        "length(data('line_hover_store')) && vlSelectionTest('line_hover_store', datum)",
        "!length(data('line_hover_store')) && vlSelectionTest('line_click_store', datum)",
    ]
)

judge_index = {judge: index for index, judge in enumerate(judge_order)}
judge_label_lines = {
    "OpenAI GPT 5.5": ["OpenAI GPT", "5.5"],
    "Claude Opus 4.8": ["Claude Opus", "4.8"],
    "Gemini 3.5 Flash": ["Gemini 3.5", "Flash"],
    "DeepSeek V4 Pro": ["DeepSeek V4", "Pro"],
}
judge_label_expr = " : ".join(
    [
        f"datum.value == {judge_index[judge]} ? {judge_label_lines.get(judge, [judge])[0:2]}"
        for judge in judge_order
    ]
    + ["''"]
)

plot_data = judge_model_scores.with_columns(
    pl.col("judge").replace(judge_index).cast(pl.Float64).alias("judge_index")
)
hover_target_rows = []
for candidate_key, candidate_data in plot_data.sort(["candidate", "judge_index"]).partition_by("candidate", as_dict=True).items():
    candidate_name = candidate_key[0] if isinstance(candidate_key, tuple) else candidate_key
    points = candidate_data.to_dicts()
    for start, end in zip(points, points[1:]):
        for step in range(18):
            t = step / 18
            hover_target_rows.append(
                {
                    "candidate": candidate_name,
                    "judge_index": start["judge_index"] + (end["judge_index"] - start["judge_index"]) * t,
                    "win_pct": start["win_pct"] + (end["win_pct"] - start["win_pct"]) * t,
                }
            )
    hover_target_rows.append(
        {
            "candidate": candidate_name,
            "judge_index": points[-1]["judge_index"],
            "win_pct": points[-1]["win_pct"],
        }
    )
hover_target_data = pl.DataFrame(hover_target_rows)
line_label_data = plot_data.filter(pl.col("judge") == judge_order[-1]).with_columns(
    (pl.col("judge_index") + 0.08).alias("label_x")
)

candidate_color = alt.Color(
    "candidate:N",
    sort=model_order,
    scale=alt.Scale(domain=model_order, range=MODEL_PALETTE),
    legend=None,
)

x_encoding = alt.X(
    "judge_index:Q",
    scale=alt.Scale(domain=[-0.05, len(judge_order) - 0.15]),
    axis=alt.Axis(
        values=list(judge_index.values()),
        labelAngle=0,
        labelExpr=judge_label_expr,
        labelLimit=90,
        title="Judge model",
    ),
)
y_encoding = alt.Y(
    "win_pct:Q",
    scale=alt.Scale(domain=[0, 100]),
    axis=alt.Axis(values=[0, 25, 50, 75, 100], labelExpr="datum.value + '%'"),
    title="Overall win percentage",
)

base = alt.Chart(plot_data).encode(
    x=x_encoding,
    y=y_encoding,
    detail="candidate:N",
)

lines = base.mark_line(point=True, strokeWidth=2.5).encode(
    color=candidate_color,
    opacity=alt.condition(active_candidate, alt.value(1), alt.value(0.14)),
)

hover_targets = alt.Chart(hover_target_data).mark_circle(size=100, opacity=0.001).encode(
    x=x_encoding,
    y=y_encoding,
    color=alt.value("#000000"),
)

line_labels = alt.Chart(line_label_data).mark_text(
    align="left",
    baseline="middle",
    dx=4,
    fontSize=11,
).encode(
    x=alt.X("label_x:Q", scale=alt.Scale(domain=[-0.05, len(judge_order) - 0.15])),
    y=y_encoding,
    text="candidate:N",
    color=candidate_color,
    opacity=alt.condition(active_candidate, alt.value(1), alt.value(0.18)),
)

(lines + hover_targets + line_labels).add_params(
    line_hover,
    line_click,
).properties(
    width=760,
    height=430,
    title="Candidate Win Percentage by Judge Model",
)


## Pairwise results table comparing judge and candidate scores

### Description

A table with the same number of rows and columns

Each row and column represents a judge model. Rows and columns are in the same order.
Values in table represent the candidate model's overall win percentage when then candidate
model is the model in the row and the judge model is the model in the column.

Color code cells to represent a high or low value.

In [ ]:
matrix_label_lines = {
    "OpenAI GPT 5.5": ["OpenAI GPT", "5.5"],
    "Claude Opus 4.8": ["Claude Opus", "4.8"],
    "Gemini 3.5 Flash": ["Gemini 3.5", "Flash"],
    "DeepSeek V4 Pro": ["DeepSeek V4", "Pro"],
}
matrix_label_expr = " : ".join(
    [
        f"datum.label == '{judge}' ? {matrix_label_lines.get(judge, [judge])[0:2]}"
        for judge in judge_order
    ]
    + ["datum.label"]
)

matrix_plot_data = judge_matrix_scores.with_columns(
    (pl.col("win_pct").round(0).cast(pl.Int64).cast(pl.Utf8) + pl.lit("%")).alias("label")
)

heatmap = alt.Chart(matrix_plot_data).mark_rect(stroke="white", strokeWidth=2).encode(
    x=alt.X(
        "judge:N",
        sort=judge_order,
        axis=alt.Axis(labelAngle=0, labelExpr=matrix_label_expr, labelLimit=90),
        title="Judge model",
    ),
    y=alt.Y(
        "candidate:N",
        sort=judge_order,
        axis=alt.Axis(labelExpr=matrix_label_expr, labelLimit=90),
        title="Candidate model",
    ),
    color=alt.Color(
        "win_pct:Q",
        scale=alt.Scale(domain=[0, 100], range=["#eff6ff", "#1d4ed8"]),
        legend=None,
    ),
)

labels = alt.Chart(matrix_plot_data).mark_text(fontSize=13, fontWeight="bold").encode(
    x=alt.X("judge:N", sort=judge_order),
    y=alt.Y("candidate:N", sort=judge_order),
    text="label:N",
    color=alt.condition(alt.datum.win_pct >= 55, alt.value("white"), alt.value(TEXT_COLOR)),
)

(heatmap + labels).properties(
    width=520,
    height=420,
    title="Judge Model Self-Comparison Matrix",
)
